# Suite P3CHK — Checking Python 3 modules against versions, and as trees

Which versions of Python 3 have which constructs, as the port records it (stage 2): `Python3.Standards` records the
versions `Programs.Python3` covers as data, each holding the statement and clause kinds it has, as mbse-expressions'
dialects record the terms' (mbse-basis CAP-268); `Python3.Check.check(store, path, version)` lists the constructs a
stored module uses that a version lacks, each located by its path, reading those standards and, for what a property's
value makes, its own `FEATURES`; `Python3.Check.validate(store, path)` lists a module's problems as a tree, each kind's
own rules and its terms' as their dialects validate them. Ported from suite PY (availability by version, 3.0 to 3.15,
and each kind's own checks) and PYPRT-06 (printing checked the version). Each case parses source with
`Python3.Parse.parse` into one store, `Python3.store()` prepared by `Standards.prepare`, or writes a tree by hand
where the parser would refuse it.

In [ ]:
import itertools
from pathlib import Path

from mbse.Expressions.Framework import Standards as ExpressionStandards
from mbse.Expressions.Python.Python3 import Expressions as P
from mbse.Programs import Python3
from mbse.Programs.Python3 import Check, Parse, Print, Standards
from mbse.Programs.Python3.Statements import KINDS, LANGUAGE
from support import raises

SOURCES = Path("../../conformance/sources")
STORE = Standards.prepare(Python3.store())
PATHS = (f"case{n}.py" for n in itertools.count())


def parsed(text):
    """The path of a new module holding `text`, parsed into the store."""
    path = next(PATHS)
    STORE.transaction(Parse.parse(text, path))
    return path


def check(text, version):
    return Check.check(STORE, parsed(text), version)


def needs(text, minor, *problems):
    """`text` is fine in Python 3.`minor`, and has `problems` in the version before."""
    path = parsed(text)
    assert Check.check(STORE, path, (3, minor)) == [], Check.check(STORE, path, (3, minor))
    assert Check.check(STORE, path, (3, minor - 1)) == list(problems), Check.check(STORE, path, (3, minor - 1))


# Trees written by hand, which the parser would refuse: a statement of a kind, its properties and its children
def S(kind, children=None, **properties):
    """A statement of `kind`, with its properties, and its children by property (a list alone its body)."""
    made = Parse._Statement("s?", kind, 1).set(**properties)
    if isinstance(children, list):  # a body
        children = {"body": children}
    for prop, statements in (children or {}).items():
        made.hold(prop, statements)
    return made


def written(*top):
    """The path of a new module whose body is `top`, written as `Parse.parse` writes one."""
    path, made = next(PATHS), []

    def collect(statement):
        made.append(statement)
        for _, _, child in statement.children:
            collect(child)

    for statement in top:
        collect(statement)
    for i, statement in enumerate(made):
        statement.symbol = f"s{i}"

    def transaction(txn):
        L = txn.Programs.Python3
        symbols = {"ws": txn.Programs.Framework.Workspace, "module": L.Module,
                   **{s.symbol: getattr(L, s.kind) for s in made}}
        return txn.uid(path).mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
            .symbols(symbols)
            .spec([
                lambda _: _.ws.files({path: lambda e: e.file(_.module)}),
                lambda _: _.module.body([lambda e, i=i, s=s: e.child(getattr(_, s.symbol)).property("body").index(i)
                                         for i, s in enumerate(top)]),
                *(Parse._spec(s) for s in made),
            ])]))])

    STORE.transaction(transaction)
    return path


def validate(*top):
    return Check.validate(STORE, written(*top))


def v(name):
    return P.name(name)


def pas():
    return [S("Pass")]


def comments():
    return [S("Comment", text=" c")]

## P3CHK-01 · The versions as data: `Programs.Python3` covers Python 3.0 to 3.15, each holding the statement kinds it has, beside mbse-expressions' dialects' own; run again, nothing changes

In [ ]:
with STORE.transaction():
    language = STORE.singleton(LANGUAGE)
    standards = ExpressionStandards.standards(language)
    assert list(standards) == [f"Python 3.{minor}" for minor in range(16)]
    assert ExpressionStandards.find(language, "Python 3.12") is standards["Python 3.12"]
    assert ExpressionStandards.properties(standards["Python 3.12"]) == {"bodies": ("org.python",),
                                                                         "revision": ("version", "3.12")}
    assert ExpressionStandards.designation_of(standards["Python 3.10"]) == "Python 3.10"

    def kinds(minor):
        return {kind.name.rsplit(".", 1)[-1] for kind in ExpressionStandards.kinds(standards[f"Python 3.{minor}"])}

    assert kinds(15) == set(KINDS) and len(KINDS) == 31
    assert set(KINDS) - kinds(0) == {"AsyncFor", "AsyncWith", "AsyncFunctionDef", "AnnAssign", "Match", "MatchCase",
                                     "TryStar", "TypeAlias"}
    assert kinds(5) - kinds(4) == {"AsyncFor", "AsyncWith", "AsyncFunctionDef"} and kinds(6) - kinds(5) == {"AnnAssign"}
    assert kinds(10) - kinds(9) == {"Match", "MatchCase"} and kinds(11) - kinds(10) == {"TryStar"}
    assert kinds(12) - kinds(11) == {"TypeAlias"} and kinds(15) == kinds(12)
    assert {minor: sorted(k for k, since in Standards.SINCE.items() if since == minor) for minor in (5, 12)} == {
        5: ["AsyncFor", "AsyncFunctionDef", "AsyncWith"], 12: ["TypeAlias"]}
    # The terms' versions are mbse-expressions' own, each dialect's by the same designations
    for dialect in Standards.DIALECTS[1:]:
        assert set(ExpressionStandards.standards(STORE.singleton(dialect))) <= set(standards), dialect
    patterns = ExpressionStandards.standards(STORE.singleton("Expressions.Python.Patterns"))
    assert list(patterns)[0] == "Python 3.10"  # patterns arrived in 3.10
    expressions = ExpressionStandards.standards(STORE.singleton("Expressions.Python.Python3"))
    tags = {kind.name for kind in ExpressionStandards.kinds(expressions["Python 3.7"])}
    assert "Expressions.Python.Python3.JoinedStr" in tags and "Expressions.Python.Python3.NamedExpr" not in tags
# The language is one of the framework's dialects, which is what may own a standard
dialects = STORE.registered("Expressions.Framework.Dialect")
assert any(b.name == "programs_python3" and b.schema.name == "Programs.Python3.Language" for b in dialects.branches)
# Run again, each writes nothing: the same versions, each with the same kinds
Standards.prepare(STORE)
with STORE.transaction():
    again = ExpressionStandards.standards(STORE.singleton(LANGUAGE))
    assert list(again) == list(standards) and len(ExpressionStandards.kinds(again["Python 3.15"])) == 31
assert Standards.designation(14) == "Python 3.14" and list(Standards.MINORS) == list(range(16))
print("ok")

## P3CHK-02 · Where kinds and features exist: since which version of Python 3, each located by its path

In [ ]:
needs("def g():\n    yield from x\n", 3, "body[0].body[0].value: yieldfrom needs Python 3.3")
needs("await a\n", 5, "body[0].value: await needs Python 3.5")
needs("a @ b\n", 5, "body[0].value: binop.op '@' needs Python 3.5")
needs("a @= b\n", 5, "body[0]: AugAssign.op '@' needs Python 3.5")
needs("async def f():\n    async with a:\n        async for x in y:\n            pass\n", 5,
      "body[0]: AsyncFunctionDef needs Python 3.5", "body[0].body[0]: AsyncWith needs Python 3.5",
      "body[0].body[0].body[0]: AsyncFor needs Python 3.5")
needs("a: int\n", 6, "body[0]: AnnAssign needs Python 3.6")
needs("f'x'\n", 6, "body[0].value: joinedstr needs Python 3.6")
needs("[x async for x in y]\n", 6, "body[0].value: comprehension.is_async needs Python 3.6")
needs("(a := b)\n", 8, "body[0].value: namedexpr needs Python 3.8")
needs("def f(a, /):\n    pass\n", 8, "body[0].args: arguments.posonlyargs needs Python 3.8")
needs("lambda a, /: a\n", 8, "body[0].value: arguments.posonlyargs needs Python 3.8")
needs("f'{a=}'\n", 8, "body[0].value: formattedvalue.debug needs Python 3.8")
needs("@d[0]\ndef f():\n    pass\n", 9, "body[0]: FunctionDef with a decorator that is not a dotted name needs Python 3.9")
needs("@(d)\nasync def f():\n    pass\n", 9,
      "body[0]: AsyncFunctionDef with a decorator that is not a dotted name needs Python 3.9")
assert check("@d\n@d.e(1)\nclass C:\n    pass\n", (3, 8)) == []
needs("match a:\n    case None:\n        pass\n", 10, "body[0]: Match needs Python 3.10",
      "body[0].cases[0]: MatchCase needs Python 3.10", "body[0].cases[0].pattern: matchsingleton needs Python 3.10")
needs("try:\n    pass\nexcept* E:\n    pass\n", 11, "body[0]: TryStar needs Python 3.11")
needs("a[*b]\n", 11, "body[0].value: subscript with an unpacked index needs Python 3.11")
needs("x = a[i, *b]\n", 11, "body[0].value: subscript with an unpacked index needs Python 3.11")
assert check("a[i,]\na[i]\n", (3, 10)) == []
needs("type X = int\n", 12, "body[0]: TypeAlias needs Python 3.12")
needs("class C[T]:\n    pass\n", 12, "body[0]: ClassDef.type_params needs Python 3.12",
      "body[0].type_params[0]: typevar needs Python 3.12")
needs("def f[T](x: T) -> T:\n    pass\n", 12, "body[0]: FunctionDef.type_params needs Python 3.12",
      "body[0].type_params[0]: typevar needs Python 3.12")
needs("type X[**P = a] = int\n", 13, "body[0].type_params[0]: paramspec.default_value needs Python 3.13")
needs("type X[*Ts = a] = int\n", 13, "body[0].type_params[0]: typevartuple.default_value needs Python 3.13")
needs("type X[T: int = str] = T\n", 13, "body[0].type_params[0]: typevar.default_value needs Python 3.13")
needs("t'x'\n", 14, "body[0].value: templatestr needs Python 3.14")
# Not found: an except clause's types written bare, which the tree does not record (Parse writes a tuple either way)
assert check("try:\n    pass\nexcept A, B:\n    pass\n", (3, 13)) == []
needs("lazy import m\n", 15, "body[0]: Import.is_lazy needs Python 3.15")
needs("lazy from . import a\n", 15, "body[0]: ImportFrom.is_lazy needs Python 3.15")
needs("[*x for x in y]\n", 15, "body[0].value: listcomp of unpacked elements needs Python 3.15")
needs("{*x for x in y}\n", 15, "body[0].value: setcomp of unpacked elements needs Python 3.15")
needs("(*x for x in y)\n", 15, "body[0].value: generatorexp of unpacked elements needs Python 3.15")
needs("{**x for x in y}\n", 15, "body[0].value: dictcomp of unpacked mappings needs Python 3.15")
assert check("{x: y for x in z}\n[x for x in y]\n", (3, 14)) == []
# A statement's problems, then its properties', then its children's; each once, in source order
assert check("x = t'a'\nif (y := t'b'):\n    lazy import m\n", (3, 7)) == [
    "body[0].value: templatestr needs Python 3.14", "body[1].test: namedexpr needs Python 3.8",
    "body[1].test: templatestr needs Python 3.14", "body[1].body[0]: Import.is_lazy needs Python 3.15"]
assert check("(a := (b := c))\n", (3, 7)) == ["body[0].value: namedexpr needs Python 3.8"]
assert check("class C(B, metaclass=M[*t]):\n    x: f'{a!r}'\n", (3, 5)) == [
    "body[0].keywords[0]: subscript with an unpacked index needs Python 3.11", "body[0].body[0]: AnnAssign needs Python 3.6",
    "body[0].body[0].annotation: joinedstr needs Python 3.6"]
print("ok")

## P3CHK-03 · The version by `(3, minor)` or by name; versions the language does not cover, a store not prepared, and a path the workspace does not hold

In [ ]:
from mbse.Programs.Framework import Model
path = parsed("type X = int\n")
assert Check.check(STORE, path, (3, 11)) == Check.check(STORE, path, "Python 3.11") == [
    "body[0]: TypeAlias needs Python 3.12"]
assert Check.check(STORE, path, (3, 12)) == Check.check(STORE, path, "Python 3.12") == []
for version in ((3, 16), (2, 7), (4, 0), "Python 4.0", "3.12"):
    with raises(LookupError, match="Programs.Python3 covers no version"):
        Check.check(STORE, path, version)
with raises(LookupError):
    Check.check(STORE, "nowhere.py", (3, 12))
with raises(LookupError):
    Check.validate(STORE, "nowhere.py")
# Within the caller's transaction, a module's visitor is read; the store and a path open their own
with STORE.transaction():
    module = STORE.singleton(Model.WORKSPACE).files({"path": path})
    assert Check.check_file(STORE, module, (3, 11)) == ["body[0]: TypeAlias needs Python 3.12"]
    assert Check.validate_file(module) == []
    with raises(ValueError, match="transactions do not nest"):
        Check.check(STORE, path, (3, 11))
# A store that holds the installs alone has no versions; one with the language's alone has no terms' versions
bare = Python3.store()
bare.transaction(Parse.parse("pass\n", "a.py"))
with raises(LookupError, match="run Standards.prepare"):
    Check.check(bare, "a.py", (3, 12))
bare.transaction(Standards.install)
bare.transaction(Standards.define)
with raises(LookupError, match="Expressions.Python.Python3 covers no version"):
    Check.check(bare, "a.py", (3, 12))
assert Check.check(Standards.prepare(bare), "a.py", (3, 0)) == []
print("ok")

## P3CHK-04 · The conformance source: none of its constructs lacking in Python 3.14, nor 3.15's added to it; in older versions, each construct where it arrived

In [ ]:
text = (SOURCES / "python314.py").read_text(encoding="utf-8")
path = parsed(text)
assert Check.check(STORE, path, (3, 14)) == Check.check(STORE, path, (3, 15)) == []
assert Check.check(STORE, path, (3, 13)) == ["body[19].body[1].value: templatestr needs Python 3.14"]
problems = Check.check(STORE, path, (3, 9))
assert all(p.endswith(("3.10", "3.11", "3.12", "3.13", "3.14")) for p in problems), problems
assert problems[:3] == ["body[9]: TypeAlias needs Python 3.12", "body[9].type_params[0]: typevar needs Python 3.12",
                        "body[9].type_params[1]: typevar needs Python 3.12"]
oldest = Check.check(STORE, path, (3, 0))
features = {p.split(": ", 1)[1].rsplit(" needs ", 1)[0] for p in oldest}
assert features >= {"TypeAlias", "TryStar", "Match", "MatchCase", "AnnAssign", "AsyncFunctionDef", "AsyncWith",
                    "AsyncFor", "await", "yieldfrom", "joinedstr", "namedexpr", "templatestr", "matchor", "typevar",
                    "paramspec", "typevartuple", "ClassDef.type_params", "binop.op '@'", "AugAssign.op '@'",
                    "arguments.posonlyargs", "comprehension.is_async", "formattedvalue.debug",
                    "subscript with an unpacked index", "typevar.default_value",
                    "ClassDef with a decorator that is not a dotted name"}, features
assert len(oldest) == len(set(oldest)) and set(problems) <= set(oldest)
# Python 3.15's constructs added to it: only they lack in 3.14
later = parsed(text + "\nlazy import json\nlazy from os import path\nflat = [*row for row in rows]\n"
               "merged = {**m for m in maps}\n")
assert Check.check(STORE, later, (3, 15)) == []
assert Check.check(STORE, later, (3, 14)) == [
    "body[25]: Import.is_lazy needs Python 3.15", "body[26]: ImportFrom.is_lazy needs Python 3.15",
    "body[27].value: listcomp of unpacked elements needs Python 3.15",
    "body[28].value: dictcomp of unpacked mappings needs Python 3.15"]
assert Check.validate(STORE, path) == [] and Check.validate(STORE, later) == []
print(len(oldest), "constructs lacking in Python 3.0")

## P3CHK-05 · Each kind's own rules: what it needs, what its blocks and lists must hold, what goes together, the names it declares, and where a statement is

In [ ]:
problems = {
    S("Assign", value=v("a")): "body[0]: an Assign needs a target",
    S("Delete"): "body[0]: a Delete needs a target",
    S("Import"): "body[0]: an Import needs a name",
    S("ImportFrom", module="m"): "body[0]: an ImportFrom needs a name",
    S("ImportFrom", names=[Parse._helper("alias", name="*")]): "body[0]: an ImportFrom needs a module or a level",
    S("Global"): "body[0]: a Global needs a name",
    S("Nonlocal"): "body[0]: a Nonlocal needs a name",
    S("Raise", cause=v("c")): "body[0]: a Raise with a cause needs an exc",
    S("Expr"): "body[0]: an Expr needs a value",
    S("Assert"): "body[0]: an Assert needs a test",
    S("If", {"body": pas()}): "body[0]: an If needs a test",
    S("If", {"body": comments()}, test=v("a")): "body[0]: an If needs a statement in its body",
    S("If", {"body": pas(), "orelse": comments()}, test=v("a")): "body[0]: an If needs a statement in its orelse",
    S("While", test=v("a")): "body[0]: a While needs a statement in its body",
    S("For", target=v("a"), iter=v("b")): "body[0]: a For needs a statement in its body",
    S("AsyncFor", target=v("a"), iter=v("b")): "body[0]: an AsyncFor needs a statement in its body",
    S("Try", {"body": pas()}): "body[0]: a Try needs a handler or a finalbody",
    S("Try", {"body": pas(), "orelse": pas(), "finalbody": pas()}): "body[0]: a Try with an orelse needs a handler",
    S("TryStar", {"body": pas()}): "body[0]: a TryStar needs a handler or a finalbody",
    S("Try", {"body": pas(), "handlers": [S("ExceptHandler")]}):
        "body[0].handlers[0]: an ExceptHandler needs a statement in its body",
    S("With", {"body": pas()}): "body[0]: a With needs an item",
    S("AsyncWith", items=[Parse._helper("withitem", context_expr=v("a"))]):
        "body[0]: an AsyncWith needs a statement in its body",
    S("Match", subject=v("a")): "body[0]: a Match needs a case",
    S("Match", {"cases": [S("MatchCase", pas())]}, subject=v("a")): "body[0].cases[0]: a MatchCase needs a pattern",
    S("FunctionDef", {"body": pas()}, args=Parse._helper("arguments")): "body[0]: a FunctionDef needs a name",
    S("AsyncFunctionDef", name="f", args=Parse._helper("arguments")):
        "body[0]: an AsyncFunctionDef needs a statement in its body",
    S("ClassDef", name="C"): "body[0]: a ClassDef needs a statement in its body",
    S("AugAssign", target=v("a"), op="@@", value=v("b")): "body[0]: AugAssign.op cannot be '@@'",
    S("TypeAlias", name=v("X")): "body[0]: a TypeAlias needs a value",
}
for statement, problem in problems.items():
    found = validate(statement)
    assert found == [problem], (statement.kind, found)
# What needs several, each
assert validate(S("AugAssign")) == ["body[0]: an AugAssign needs a target", "body[0]: an AugAssign needs an op",
                                    "body[0]: an AugAssign needs a value"]
assert validate(S("For", pas())) == ["body[0]: a For needs a target", "body[0]: a For needs an iter"]
# Names a statement declares: identifiers, not keywords; dotted names; `*` only in a from-import
assert validate(S("FunctionDef", pas(), name="class", args=Parse._helper("arguments"))) == [
    "body[0].name: 'class' is a keyword"]
assert validate(S("ClassDef", pas(), name="1x")) == ["body[0].name: '1x' is not an identifier"]
assert validate(S("Global", names=["a", "None", "b-c"])) == ["body[0].names[1]: 'None' is a keyword",
                                                             "body[0].names[2]: 'b-c' is not an identifier"]
assert validate(S("Try", {"body": pas(), "handlers": [S("ExceptHandler", pas(), name="for")]})) == [
    "body[0].handlers[0].name: 'for' is a keyword"]
assert validate(S("Import", names=[Parse._helper("alias", name="a.b", asname="if"),
                                   Parse._helper("alias", name="a..b"), Parse._helper("alias", name="*"),
                                   Parse._helper("alias", asname="x")])) == [
    "body[0].names[0]: 'if' is a keyword", "body[0].names[1]: '' is not an identifier",
    "body[0].names[2]: '*' is not an identifier", "body[0].names[3]: an alias needs a name"]
assert validate(S("ImportFrom", module="a.class", names=[Parse._helper("alias", name="*")])) == [
    "body[0].module: 'class' is a keyword"]
for spelling in ("x_1", "Größe", "_", "match", "case", "type", "lazy", "print"):  # soft keywords are identifiers
    assert validate(S("Nonlocal", names=[spelling])) == [], spelling
# Where a statement is: in its parent's blocks, a clause only in its own place
assert validate(S("ExceptHandler", pas())) == ["body[0]: a Module's body cannot hold an ExceptHandler"]
assert validate(S("Try", {"body": pas(), "handlers": [S("Pass")]})) == [
    "body[0].handlers[0]: a Try's handlers cannot hold a Pass"]
assert validate(S("If", {"body": pas(), "cases": [S("MatchCase", pas(), pattern=Parse._term("matchas"))]},
                  test=v("a"))) == ["body[0].cases[0]: an If's cases cannot hold a MatchCase"]
assert validate(S("Match", {"cases": [S("MatchCase", pas(), pattern=Parse._term("matchas"))], "body": pas()},
                  subject=v("a"))) == ["body[0].body[0]: a Match's body cannot hold a Pass"]
assert validate(S("Pass", {"body": pas()})) == ["body[0].body[0]: a Pass's body cannot hold a Pass"]
# What is fine
fine = [
    S("Try", {"body": pas(), "handlers": [S("ExceptHandler", pas())], "orelse": pas()}),
    S("Try", {"body": pas(), "finalbody": comments() + pas()}),
    S("ImportFrom", level=1, names=[Parse._helper("alias", name="*")]),
    S("Raise"), S("Return"), S("Comment", text=" c"),
]
assert validate(*fine) == []
print(len(problems), "problems")

## P3CHK-06 · Terms as their dialects validate them, located by the property that holds them; names they refer to taken as bound

In [ ]:
assert validate(S("Expr", value=P.set_())) == ["body[0].value: a set needs an element"]
assert validate(S("Expr", value=P.boolop("and", v("a")))) == ["body[0].value: a boolop needs two values"]
lambda_ = P.lambda_(v("a"), [P.arg("a", annotation=v("int"))])
assert validate(S("Expr", value=lambda_)) == ["body[0].value: a lambda's parameters have no annotations"]
one = Parse._term("matchor", patterns=[Parse._term("matchas")])
assert validate(S("Match", {"cases": [S("MatchCase", pas(), pattern=one)]}, subject=v("a"))) == [
    "body[0].cases[0].pattern: a MatchOr needs two patterns"]
assert validate(S("Assign", targets=[v("a"), P.set_()], value=v("undefined"))) == [
    "body[0].targets[1]: a set needs an element"]
arguments = Parse._helper("arguments", args=[P.arg("a", default_value=P.set_())])
assert validate(S("FunctionDef", pas(), name="f", args=arguments)) == ["body[0].args: a set needs an element"]
# A whole module, parsed: none
assert Check.validate(STORE, parsed("def f(a, *, b=1):\n    return [x for x in a if (y := x)]\n")) == []
print("ok")

## P3CHK-07 · Printing writes what the store holds; checking it against a version is `check`'s, and validating it `validate`'s (ported from PYPRT-06)

In [ ]:
path = parsed("t'x'\n")
assert Print.print_module(STORE, path) == "t'x'\n"  # printed, whatever the version
assert Check.check(STORE, path, (3, 12)) == ["body[0].value: templatestr needs Python 3.14"]
path = parsed("lazy import a\n")
assert Print.print_module(STORE, path) == "lazy import a\n"
assert Check.check(STORE, path, (3, 14)) == ["body[0]: Import.is_lazy needs Python 3.15"]
path = parsed("(x := a)\nreturn (x := a)\nfor x in (x := a):\n    pass\n")
assert Check.check(STORE, path, (3, 12)) == [] and Check.validate(STORE, path) == []
assert Print.print_module(STORE, path) == "(x := a)\nreturn (x := a)\nfor x in (x := a):\n    pass\n"
assert validate(S("If", pas())) == ["body[0]: an If needs a test"]
assert validate(S("FunctionDef", pas(), name="class", args=Parse._helper("arguments"))) == [
    "body[0].name: 'class' is a keyword"]
print("ok")